<a href="https://colab.research.google.com/github/AlexKitipov/Aether_OS_Nexus_Core_v.0.3/blob/main/AeterOS__Compiling_v_0_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1 — Install Rust nightly and required components:

In [ ]:
import os

# Install rustup
get_ipython().system('curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y')

# Add cargo to the current session's PATH
cargo_bin_path = os.path.expanduser('~/.cargo/bin')
if cargo_bin_path not in os.environ['PATH']:
    os.environ['PATH'] = f"{cargo_bin_path}:{os.environ['PATH']}"
    print(f"Added {cargo_bin_path} to PATH for current session.")

# Now rustup should be available
get_ipython().system('rustup toolchain install nightly')
get_ipython().system('rustup default nightly')
get_ipython().system('rustup component add rust-src --toolchain nightly')

## STEP 2 — Install LLVM and QEMU:

## STEP 2.1 — Re-install LLVM and QEMU as requested:

In [ ]:
get_ipython().system('apt-get update')
get_ipython().system('apt-get install -y qemu-system-x86 llvm clang lld')

In [ ]:
get_ipython().system('apt-get update')
get_ipython().system('apt-get install -y qemu-system-x86 llvm clang lld')

## STEP 3 — Clone the repository CLEANLY:

In [ ]:
get_ipython().system('rm -rf Aether_OS_Nexus_Core_v.0.3')
get_ipython().system('git clone https://github.com/AlexKitipov/Aether_OS_Nexus_Core_v.0.3.git')

## STEP 4 — Enter the project:

In [ ]:
get_ipython().run_line_magic('cd', '/content/Aether_OS_Nexus_Core_v.0.3/AetherOS')

## STEP 5 — Clean previous builds:

In [ ]:
get_ipython().system('cargo clean')

## STEP 6 — Build ONLY the kernel crate (not the workspace):

In [ ]:
get_ipython().system('cargo build -p aetheros-kernel --release')

In [ ]:
get_ipython().system('cargo build --release')

## STEP 6.1 — Build the entire workspace:

In [ ]:
get_ipython().system('cargo build --workspace --release')

## STEP 7 — Print the REAL errors without modifying anything:

## STEP 8 — Build the bootable kernel image (bootloader_api 0.11 flow):

`cargo bootimage` is intentionally not used for this repository. The kernel binary can exist at `target/x86_64-unknown-none/release/aetheros-kernel`, but legacy `bootimage` discovers kernels only from executables reported by the specific Cargo JSON build it launches and then expects the older bootloader metadata flow. With `bootloader_api`/`bootloader` 0.11, use the repository image builder instead.


In [ ]:
# No bootimage install is required. Build the kernel ELF and wrap it with the repo image builder.
get_ipython().system('BOOT_MODE=uefi ./scripts/build_kernel_image.sh')

In [ ]:
# Optional: confirm the kernel ELF that the image builder consumes.
get_ipython().system('file target/x86_64-unknown-none/release/aetheros-kernel')
get_ipython().system('ls -lh target/x86_64-unknown-none/release/aetheros-uefi.img')

## STEP 9 — Check the environment setup

In [ ]:
get_ipython().system('pwd')
get_ipython().system('ls -l scripts/')
get_ipython().system('./scripts/check_env.sh')

## STEP 10 — Run the Kernel in QEMU

In [ ]:
get_ipython().system('./scripts/run_qemu.sh')

## STEP 11 — Install OVMF for UEFI boot

In [ ]:
get_ipython().system('apt-get update')
get_ipython().system('apt-get install -y ovmf')

# Find the OVMF_CODE.fd file after installation
import os
ovmf_path = ''
for root, dirs, files in os.walk('/usr/share/OVMF/'):
    for file in files:
        if file.endswith('OVMF_CODE.fd'):
            ovmf_path = os.path.join(root, file)
            break
    if ovmf_path: break

if ovmf_path:
    os.environ['OVMF_CODE'] = ovmf_path
    print(f'OVMF_CODE set to: {os.environ["OVMF_CODE"]}')
else:
    print('Error: OVMF_CODE.fd not found after installation.')


## STEP 12 — Re-run the Kernel in QEMU

In [ ]:
get_ipython().system('./scripts/run_qemu.sh')

In [ ]:
get_ipython().system('cat << \'EOF\' > scripts/run_qemu.sh\n#!/usr/bin/env bash\nset -euo pipefail\n\nROOT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")/.." && pwd)"\nBOOT_MODE="${BOOT_MODE:-uefi}"\nBIOS_IMAGE="${ROOT_DIR}/target/x86_64-unknown-none/release/aetheros-bios.img"\nUEFI_IMAGE="${ROOT_DIR}/target/x86_64-unknown-none/release/aetheros-uefi.img"\n\nif ! command -v qemu-system-x86_64 >/dev/null 2>&1; then\n  echo "[run_qemu] ERROR: qemu-system-x86_64 is not installed" >&2\n  exit 1\nfi\n\ncase "${BOOT_MODE}" in\n  bios)\n    if [[ ! -f "${BIOS_IMAGE}" ]]; then\n      echo "[run_qemu] ERROR: BIOS disk image not found at ${BIOS_IMAGE}" >&2\n      echo "[run_qemu] Hint: run BOOT_MODE=bios ./scripts/build_kernel_image.sh first." >&2\n      exit 1\n    fi\n\n    exec qemu-system-x86_64 \\\n      -drive "format=raw,file=${BIOS_IMAGE}" \\\n      -serial stdio \\\n      -no-reboot \\\n      -d int\n    ;;\n  uefi)\n    if [[ ! -f "${UEFI_IMAGE}" ]]; then\n      echo "[run_qemu] ERROR: UEFI disk image not found at ${UEFI_IMAGE}" >&2\n      echo "[run_qemu] Hint: run BOOT_MODE=uefi ./scripts/build_kernel_image.sh first." >&2\n      exit 1\n    fi\n\n    if [[ -z "${OVMF_CODE:-}" ]]; then\n      echo "[run_qemu] ERROR: OVMF_CODE must point to an OVMF_CODE.fd firmware file for UEFI boot." >&2\n      exit 1\n    fi\n\n    exec qemu-system-x86_64 \\\n      -drive "if=pflash,format=raw,readonly=on,file=${OVMF_CODE}" \\\n      -drive "format=raw,file=${UEFI_IMAGE}" \\\n      -serial stdio \\\n      -no-reboot \\\n      -d int \\\n      -display none\n    ;;\n  both)\n    echo "[run_qemu] ERROR: BOOT_MODE=both is only valid for image creation; choose bios or uefi to run." >&2\n    exit 1\n    ;;\n  *)\n    echo "[run_qemu] ERROR: BOOT_MODE must be one of: bios, uefi" >&2\n    exit 1\n    ;;\nesac\nEOF')